---
# Cuaderno 1: Mutaciones en Cáncer
---
Author: Melissa Ormeño; Alexis Salas

Laboratory: Cancer Molecular Dynamics

Date: Mayo 2026

Aims:


In [1]:
import gzip
import io
import json
import pandas as pd
import requests

In [21]:


# 1. Genes de interés
lista_genes = ['BMPR1B', 'ACVR2A', 'BMPR1A', 'ACVR1B', 'TGFBR1', 'ACVRL1','TGFBR2', 'ACVR1', 'ACVR2B', 'BMPR2', 'AMHR2', 'ACVR1C']

# 2. Proyectos del GDC para tus tejidos de interés
proyectos_gdc = [
    "TCGA-BRCA",
    "TCGA-LUAD",
    "TCGA-LUSC",
    "TCGA-KIRC",
    "TCGA-KIRP",
    "TCGA-KICH",
    "TCGA-BLCA",
    "TCGA-COAD",
    "TCGA-READ",
    "TCGA-STAD",
    "TCGA-LGG",
    "TCGA-GBM",
    "TCGA-UCEC",
    "TCGA-UCS",
    "TCGA-OV",
    "TCGA-LIHC",
    "TCGA-PAAD",
    "TCGA-ESCA",
    "TCGA-THCA",
    "TCGA-SKCM",
    "TARGET-RT",
    "TARGET-WT",
]

ssm_endpt = "https://api.gdc.cancer.gov/ssms"
lista_mutaciones = []

print(
    f"Iniciando consulta directa a la base de datos de mutaciones para {len(lista_genes)} genes en {len(proyectos_gdc)} proyectos...\n"
)

# Iteramos proyecto por proyecto para evitar saturar la API
for idx, proyecto in enumerate(proyectos_gdc, 1):

    # Filtro de la base de datos: Queremos el gen X AND el proyecto Y
    filters = {
        "op": "and",
        "content": [
            {"op": "in", "content": {"field": "consequence.transcript.gene.symbol", "value": lista_genes}},
            {"op": "in", "content": {"field": "occurrence.case.project.project_id", "value": [proyecto]}},
        ],
    }

    # Campos exactos que me pediste extraer (Alelos, IDs, Aminoácidos, etc.)
    fields = [
        "ssm_id",
        "chromosome",
        "genomic_dna_change",  # Cambio nucleótido (HGVSc aproximado)
        "mutation_subtype",  # Tipo: SNP, INS, DEL
        "ncbi_build",
        "reference_allele",  # Alelo de Referencia
        "tumor_allele",  # Alelo Mutado
        "consequence.transcript.gene.symbol",
        "consequence.transcript.aa_change",  # Cambio Aminoácido (p.R432H)
        "consequence.transcript.hgvsp",  
        "consequence.transcript.annotation.vep_impact",
        "consequence.transcript.consequence_type",  # Efecto: missense, frameshift, etc.
        "occurrence.case.submitter_id",  # ID DEL PACIENTE (ej: TCGA-XX-XXXX)
        "occurrence.case.project.project_id",
        "occurrence.case.project.primary_site",
    ]

    params = {
        "filters": json.dumps(filters),
        "fields": ",".join(fields),
        "format": "JSON",
        "size": "2000",  # Traer hasta 2000 mutaciones por proyecto (más que suficiente para un par de genes)
    }

    try:
        response = requests.get(ssm_endpt, params=params)
        data = response.json()
        hits = data.get("data", {}).get("hits", [])
    except Exception as e:
        print(f"   Error de conexión con el proyecto {proyecto}: {e}")
        continue

    if not hits:
        continue

    # Desempaquetamos la estructura JSON jerárquica del GDC a una tabla plana
    for hit in hits:
        gene_symbol = hit.get("consequence", [{}])[0].get("transcript", {}).get("gene", {}).get("symbol", "Unknown")
        aa_change = hit.get("consequence", [{}])[0].get("transcript", {}).get("aa_change", "p.?")
        transcript_info = hit.get("consequence", [{}])[0].get("transcript", {})
        aa_long = transcript_info.get("hgvsp", "p.?")
       
        # El formato MAF prefiere mayúsculas y guiones bajos (ej: Missense_Mutation en lugar de missense_variant)
        raw_efecto = hit.get("consequence", [{}])[0].get("transcript", {}).get("consequence_type", "Unknown")
        efecto_maf = raw_efecto.replace("_variant", "_Mutation").capitalize()
        if "missense" in raw_efecto:
            efecto_maf = "Missense_Mutation"
        elif "synonymous" in raw_efecto:
            efecto_maf = "Silent"
        elif "stop_gained" in raw_efecto:
            efecto_maf = "Nonsense_Mutation"
        # Un mismo cambio puede reportarse en múltiples muestras/pacientes de ese proyecto
        for occ in hit.get("occurrence", []):
            paciente_id = occ.get("case", {}).get("submitter_id", "Unknown")
            proj_id = occ.get("case", {}).get("project", {}).get("project_id", proyecto)
            tejido = occ.get("case", {}).get("project", {}).get("primary_site", "Unknown")

            registro = {
                "Patient_ID": paciente_id,
                "Tumor_Sample_Barcode": f"{paciente_id}-TUMOR", # ID de muestra aproximado
                "Hugo_Symbol": gene_symbol,
                "Chromosome": hit.get("chromosome"),
                "Variant_Classification": efecto_maf,
                "Variant_Type": hit.get("mutation_subtype"),
                "Reference_Allele": hit.get("reference_allele"),
                "Tumor_Seq_Allele2": hit.get("tumor_allele"),
                "HGVSc": hit.get("genomic_dna_change"),
                "HGVSp_Short": aa_change,
                "HGVSp": aa_long,  
                "Primary_Site": tejido,
                "Project_ID": proj_id,
            }
            lista_mutaciones.append(registro)

# 3. Consolidación de los resultados en tu DataFrame final
if lista_mutaciones:
    maf_genes = pd.DataFrame(lista_mutaciones)

    print("Completado")

    # Guardar a CSV inmediatamente
    file_path = "../data/TCGA/mutaciones_TCGA.csv"
    maf_genes.to_csv(file_path, index=False)
    print(f"Archivo guardado en: {file_path}")

Iniciando consulta directa a la base de datos de mutaciones para 12 genes en 22 proyectos...

Completado
Archivo guardado en: ../data/TCGA/mutaciones_TCGA.csv


In [12]:
maf_genes

,Patient_ID,Tumor_Sample_Barcode,Hugo_Symbol,Variant_Classification,Variant_Type,Reference_Allele,Tumor_Seq_Allele2,HGVSc,HGVSp_Short,Primary_Site,Project_ID
0,TCGA-A2-A0D0,TCGA-A2-A0D0-TUMOR,ACVR1,Missense_Mutation,Single base substitution,C,T,chr2:g.157780475C>T,V65I,[Breast],TCGA-BRCA
1,100003304,100003304-TUMOR,ACVR1,Missense_Mutation,Single base substitution,C,T,chr2:g.157780475C>T,V65I,"[Colon, Ovary, Breast, Retroperitoneum and per...",CPTAC-2
2,C3N-00321,C3N-00321-TUMOR,ACVR1,Missense_Mutation,Single base substitution,C,T,chr2:g.157780475C>T,V65I,"[Other and ill-defined sites, Other and unspec...",CPTAC-3
3,TCGA-55-7815,TCGA-55-7815-TUMOR,ACVR1,Missense_Mutation,Single base substitution,C,A,chr2:g.157780519C>A,G50V,[Bronchus and lung],TCGA-LUAD
4,TCGA-78-8660,TCGA-78-8660-TUMOR,ACVR1,Missense_Mutation,Single base substitution,A,G,chr2:g.157770421A>G,F246S,[Bronchus and lung],TCGA-LUAD
...,...,...,...,...,...,...,...,...,...,...,...
125,TCGA-ER-A194,TCGA-ER-A194-TUMOR,ACVR1,Missense_Mutation,Single base substitution,G,A,chr2:g.157799445G>A,P17S,"[Colon, Spinal cord, cranial nerves, and other...",TCGA-SKCM
126,TCGA-EE-A3J4,TCGA-EE-A3J4-TUMOR,ACVR1,Missense_Mutation,Single base substitution,G,A,chr2:g.157774162G>A,S190L,"[Colon, Spinal cord, cranial nerves, and other...",TCGA-SKCM
127,TCGA-GF-A6C9,TCGA-GF-A6C9-TUMOR,ACVR1,Silent,Single base substitution,C,T,chr2:g.157780440C>T,Q76=,"[Colon, Spinal cord, cranial nerves, and other...",TCGA-SKCM
128,TCGA-GN-A8LL,TCGA-GN-A8LL-TUMOR,ACVR1,Missense_Mutation,Single base substitution,A,G,chr2:g.157737604A>G,L486P,"[Colon, Spinal cord, cranial nerves, and other...",TCGA-SKCM


In [ ]:
ocravat_file = df_final[[
                          "chromosome", "start_position", 
                         "reference_allele",
                        "tumor_allele", "aa_mutation","gene_symbol", "project_id"]]
ocravat_file = ocravat_file.drop_duplicates()
ocravat_file = ocravat_file.reset_index()
ocravat_file.to_csv("glut1_mutation.txt", sep="\t", header=False, index=True)
ocravat_file

In [ ]:
import openpyxl


In [ ]:
import pandas as pd

df = pd.read_excel('../data/open_cravat/clinvar/anotacion_clinvar.xlsx', sheet_name='Variant',  skiprows=1)
df = df.dropna(axis=1, how="all")
df.columns

In [ ]:
df

In [ ]:
df_clinvar = df[['UID', "Samples","Position","Chrom",'Coding','Gene','Sequence Ontology', 
                 'cDNA change',
                'Protein Change', 'End Position', 'Clinical Significance', 
                'Disease Names','Allele Origin', "Variant Type", 
                 'Germline or Somatic', "Reference allele", "Alternate allele"]]
print(df_clinvar.isnull().sum())
df_clinvar

In [ ]:
df_clinvar["Disease Names"] = df_clinvar["Disease Names"].str.split("|")
df_clinvar = df_clinvar.explode("Disease Names")
df_clinvar.dropna(subset="Disease Names", inplace=True)
df_clinvar

In [ ]:
df_clinvar.to_csv("mut_glut1_clinvar.tsv", sep="\t")

In [ ]:
uniprot_ids = ['P11166']

features_dict = []

for uniprot_id in uniprot_ids:
        url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}"
        response = requests.get(url)
    
        if response.status_code != 200:
            print(f"Error al obtener {uniprot_id}: {response.status_code}")
            continue
    
        json_data = response.json()
        gen_name = (json_data.get("genes",[{}])[0].get("geneName",{}) or {}).get("value", uniprot_id)
        features = json_data.get("features", [])
        
        cross_references = json_data.get("uniProtKBCrossReferences", [])
        resultados = []
        
        for cross in cross_references:
            if cross.get("database") == "Orphanet":
                # Extraer el ID principal
                orphan_code = cross.get("id")
                
                # Convertir las propiedades en un diccionario para búsqueda rápida
                props = {p.get("key"): p.get("value") for p in cross.get("properties", [])}
                
                # Guardar en la lista de resultados
                resultados.append({
                    "Database": "Orphanet",
                    "OrphanetCode": orphan_code,
                    "DiseaseName": props.get("Disease", "N/A"),
                })
        
        # Crear el DataFrame
        df_orphan = pd.DataFrame(resultados)
        
        # Mostrar la tabla
        print(df_orphan.to_string(index=False))

In [ ]:
df_orphan

In [ ]:
from difflib import get_close_matches

enf_raras = list(df_orphan["DiseaseName"])
tabla_Datos = list(df_clinvar["Disease Names"])

# n=3 (máximo de resultados), cutoff=0.6 (60% de similitud mínima)
coincidencias = get_close_matches(tabla_Datos, enf_raras, n=1, cutoff=0.8)

print(coincidencias) # Resultado: ['SLC2A1']

In [ ]:
from difflib import get_close_matches

# Lista de referencia (Orphanet)
enf_raras = df_orphan["DiseaseName"].tolist()

# Función para buscar la mejor coincidencia
def buscar_match(enfermedad):
    # n=1 para que solo traiga la mejor, cutoff=0.6 es más flexible para nombres largos
    match = get_close_matches(enfermedad, enf_raras, n=1, cutoff=0.4)
    return match[0] if match else "Sin coincidencias"

# Aplicar la función a la columna de ClinVar
df_clinvar["Orphanet_Match"] = df_clinvar["Disease Names"].apply(buscar_match)

# Ver resultados
print(df_clinvar[["Disease Names", "Orphanet_Match"]].dropna().head())

In [ ]:
df_orphan["DiseaseName"].unique()

In [ ]:
df_clinvar["Disease Names"].unique()

In [ ]:
df_clinvar["Orphanet_Match"].unique()

In [ ]:
df_clinvar = df_clinvar.loc[df_clinvar["Orphanet_Match"] != "Sin coincidencias"]
df_clinvar.to_csv("mut_rdiseases_glut1.tsv", sep="\t", index=False)